# Retouche — validación de los dos mejores casos con tres semillas

Lee el log del experimento, elige dos configuraciones distintas con su mejor época **entrenada** y ejecuta tres semillas por caso. Solo cambian la inicialización del adaptador, el shuffle de batches y sus particiones contexto/consulta. Split, undersampling de training/validación y preprocesamiento permanecen fijos.

Cada ejecución se evalúa una sola vez al terminar, sobre la misma validación histórica del log. Se guardan probabilidades, ganancias por corte y resultados de cada ejecución. El gráfico promedia **ganancias individuales**, no probabilidades. No se evalúa 202106, no se agrega época 0 y no se guardan pesos.

Usar el mismo entorno CUDA de `BASELINE_TABICL_RETOUCHE.ipynb`, con TabICL 2.2.0. Esta notebook usa el `config_tfm.yaml` existente para directorios y la lista de semillas; no ejecuta búsqueda, producción ni runner. Los HP y la configuración de datos/entrenamiento se recuperan del log.


## 1. Archivos y alcance de la comparación


In [1]:
from pathlib import Path

ARCHIVO_CONFIG = Path("config_tfm.yaml").resolve()
NOMBRE_LOG = "Grid_log.txt"
NOMBRE_PROBABILIDADES = "validacion_probabilidades_semillas.csv"
NOMBRE_RESULTADOS = "validacion_resultados_semillas.csv"
NOMBRE_GANANCIAS = "validacion_ganancias_semillas.csv"
NOMBRE_PROMEDIOS = "validacion_ganancias_promedio.csv"

N_CASOS = 2
N_SEMILLAS = 3


## 2. Log, casos y semillas


In [2]:
import os
import gc
import json
import math
import random
import tempfile
import time
from contextlib import ExitStack
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from collections import OrderedDict

import yaml
import numpy as np
import pandas as pd


In [3]:
with ARCHIVO_CONFIG.open(encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

EXP_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["exp"] / config["exp_name"]).resolve()
DATA_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["datasets"]).resolve()
ARCHIVO_LOG = EXP_DIR / NOMBRE_LOG
ARCHIVO_PROBABILIDADES = EXP_DIR / NOMBRE_PROBABILIDADES
ARCHIVO_RESULTADOS = EXP_DIR / NOMBRE_RESULTADOS
ARCHIVO_GANANCIAS = EXP_DIR / NOMBRE_GANANCIAS
ARCHIVO_PROMEDIOS = EXP_DIR / NOMBRE_PROMEDIOS
ARCHIVOS_SALIDA = [ARCHIVO_PROBABILIDADES, ARCHIVO_RESULTADOS, ARCHIVO_GANANCIAS, ARCHIVO_PROMEDIOS]
assert all(p.parent == EXP_DIR for p in ARCHIVOS_SALIDA), "Las salidas deben quedar en el directorio del experimento."
assert len(set(ARCHIVOS_SALIDA)) == len(ARCHIVOS_SALIDA)
assert ARCHIVO_LOG not in ARCHIVOS_SALIDA

tb_log = pd.read_csv(ARCHIVO_LOG, sep="\t")
tb_log = tb_log.loc[(tb_log["epoch"] > 0) & np.isfinite(tb_log["metrica"])].copy()
tb_log["parametros_caso"] = tb_log["parametros"].map(json.loads)
tb_log["clave_hp"] = tb_log["parametros_caso"].map(
    lambda p: json.dumps(p["hp"], sort_keys=True)
)
# Primero la mejor época de cada configuración; luego las dos mejores configuraciones.
tb_casos = (tb_log.sort_values(["metrica", "epoch", "iter"], ascending=[False, True, True])
            .drop_duplicates("clave_hp").head(N_CASOS).copy())
assert len(tb_casos) == N_CASOS, "El log no contiene suficientes configuraciones entrenadas."
tb_casos.insert(0, "modelo", [f"M{i}" for i in range(1, N_CASOS + 1)])
CASOS = []
for _, fila in tb_casos.iterrows():
    parametros = deepcopy(fila["parametros_caso"])
    assert parametros["fijos"]["metrica_seleccion"] == "auc"
    assert int(fila["epoch"]) <= parametros["fijos"]["epochs"]
    CASOS.append({"modelo": fila["modelo"], "iter_log": int(fila["iter"]),
                  "epoch": int(fila["epoch"]), "auc_log": float(fila["metrica"]),
                  "fecha_log": fila["fecha"], "semilla_datos": int(fila["semilla"]),
                  "parametros": parametros})

PARAM = deepcopy(CASOS[0]["parametros"])
SEED = CASOS[0]["semilla_datos"]
for caso in CASOS:
    assert caso["semilla_datos"] == SEED, "Los casos tienen distintos splits de origen."
    assert caso["parametros"]["training"] == PARAM["training"], "Los casos tienen distintos datos de origen."
    assert caso["parametros"]["ejecucion"] == PARAM["ejecucion"], "Los casos tienen distinta configuración de ejecución."
    assert caso["parametros"]["fijos"] == PARAM["fijos"], "Los casos tienen distintos parámetros fijos."

SEMILLAS = [int(s) for s in config["training"]["semillas"][:N_SEMILLAS]]
assert len(SEMILLAS) == len(set(SEMILLAS)) == N_SEMILLAS
assert DATA_DIR.is_dir(), DATA_DIR
print("Log:", ARCHIVO_LOG)
print("Semilla de datos fija:", SEED, "| semillas de inicialización y batches:", SEMILLAS)
display(pd.DataFrame([{"modelo": c["modelo"], "epoch": c["epoch"],
    "horizonte_scheduler": c["parametros"]["fijos"]["epochs"], "auc_log": c["auc_log"],
    **c["parametros"]["hp"]} for c in CASOS]))


Log: /home/marco/code/DMEyF/exp/TabICL_Retouche/Grid_log.txt
Semilla de datos fija: 230047 | semillas de inicialización y batches: [230047, 230059, 230063]


,modelo,epoch,horizonte_scheduler,auc_log,lr_maximo,factor_lr_compuerta,compuerta_inicial,batchnorm
0,M1,10,30,0.941781,0.0001,3.0,0.02,True
1,M2,30,30,0.941609,0.0005,3.0,0.02,True


## 3. CUDA


In [4]:
# Configurar antes de importar PyTorch e inicializar CUDA.
ejecucion = PARAM["ejecucion"]
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ejecucion["cublas_workspace_config"]
for nombre in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[nombre] = str(ejecucion["hilos_cpu"])

import torch
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tabicl import FinetunedTabICLClassifier
from tabicl._finetune.data import MetaBatch, _split_ctx_query, move_meta_batch
from tabicl._sklearn.preprocessing import TransformToNumerical, EnsembleGenerator

DEVICE = torch.device(ejecucion["dispositivo"])
assert torch.cuda.is_available(), "Seleccionar el entorno PyTorch CUDA."
assert ejecucion["precision_evaluacion"] == ejecucion["precision_adaptador"] == "fp32"
torch.cuda.set_device(DEVICE)
torch.set_num_threads(ejecucion["hilos_cpu"])
torch.backends.cuda.matmul.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.deterministic = ejecucion["cudnn_deterministic"]
torch.use_deterministic_algorithms(
    ejecucion["algoritmos_deterministas"], warn_only=ejecucion["determinismo_warn_only"]
)

def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)

fijar_semilla(SEED)
print(torch.cuda.get_device_name(DEVICE))


[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.5.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NVIDIA GeForce RTX 3090


## 4. Dataset y baseline

Mismos lags y deltas del baseline. Solo se conserva historia hasta el último mes histórico definido en el log; los meses posteriores no intervienen en la comparación. Esto no cambia los valores históricos de los lags, que se calculan hacia atrás.


In [5]:
historia = pd.read_csv(DATA_DIR / PARAM["training"]["dataset"], low_memory=False)
historia = historia.loc[historia["foto_mes"] <= max(PARAM["training"]["train"])]
historia = historia.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
campos_lag = [c for c in historia.select_dtypes(include="number").columns
              if c not in PARAM["training"]["campos_excluidos"]]

# Variable numérica toda cero y sin NA en un mes: convertir esos valores a NA.
for campo in campos_lag:
    todo_cero = historia[campo].eq(0).groupby(historia["foto_mes"]).all()
    historia.loc[historia["foto_mes"].isin(todo_cero.index[todo_cero]), campo] = np.nan

print("Historial:", historia.shape)


Historial: (4050618, 155)


In [6]:
# Materializar una sola matriz CPU; conservar el baseline completo en RAM.
campos_base = [c for c in historia.columns
               if c not in {"clase_ternaria", "clase01", "fold", "azar", "training"}]
columnas = (campos_base + [f"{c}_lag1" for c in campos_lag]
            + [f"{c}_lag2" for c in campos_lag]
            + [f"{c}_delta{k}" for c in campos_lag for k in (1, 2)])
X_baseline = np.empty((len(historia), len(columnas)), dtype=np.float64, order="F")
posiciones = {c: i for i, c in enumerate(columnas)}
for campo in campos_base:
    X_baseline[:, posiciones[campo]] = historia[campo].to_numpy(dtype=np.float64)
for campo in campos_lag:
    actual = historia[campo].to_numpy(dtype=np.float64)
    grupo = historia.groupby("numero_de_cliente", sort=False)[campo]
    for k in (1, 2):
        lag = grupo.shift(k).to_numpy(dtype=np.float64)
        X_baseline[:, posiciones[f"{campo}_lag{k}"]] = lag
        X_baseline[:, posiciones[f"{campo}_delta{k}"]] = actual - lag

metadatos = historia[["numero_de_cliente", "foto_mes", "clase_ternaria"]].copy()
metadatos["clase_ternaria"] = metadatos["clase_ternaria"].astype("category")
metadatos["clase01"] = metadatos["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int64)
dataset_baseline = pd.DataFrame(X_baseline, columns=columnas, copy=False)
del historia, grupo, lag, actual
gc.collect()
print("Baseline completo en RAM:", dataset_baseline.shape)
print("Columnas para entrenar:", X_baseline.shape[1], "(referencia: 762)")

assert X_baseline.shape[1] == 762, "Revisar las columnas del baseline."


Baseline completo en RAM: (4050618, 762)
Columnas para entrenar: 762 (referencia: 762)


## 5. Split y preprocesamiento fijos

Se repite el split estratificado del log y el mismo undersampling de CONTINUA en ambos conjuntos, con la semilla de datos original. Se ajusta una sola vez el preprocesamiento de la API, exclusivamente con training. **Estas matrices y su orden se reutilizan para las seis ejecuciones.**

La ganancia se calcula sobre esta muestra fija de validación, sin ponderar ni extrapolar a otro conjunto.


In [7]:
def seleccionar_training(indices, semilla, training):
    # Mismo criterio de undersampling que LGBM/XGB: todas las bajas y 10% de CONTINUA.
    azar = np.random.default_rng(semilla).random(len(indices))
    bajas = metadatos.loc[indices, "clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).to_numpy()
    return indices[(azar <= training["undersampling"]) | bajas]


In [8]:
indices = metadatos.index[metadatos["foto_mes"].isin(PARAM["training"]["train"])].to_numpy()
idx_train, idx_valid = train_test_split(
    indices, train_size=PARAM["training"]["training_pct"] / 100,
    stratify=metadatos.loc[indices, "clase_ternaria"], random_state=SEED,
)
idx_train = seleccionar_training(idx_train, SEED, PARAM["training"])
# Una muestra fija del 30%: mismo factor de undersampling que en training.
idx_valid = seleccionar_training(idx_valid, SEED, PARAM["training"])
assert not np.intersect1d(idx_train, idx_valid).size
assert not set(PARAM["training"]["train"]) & set(PARAM["training"]["future"])
print("Train con undersampling:", len(idx_train))
print("Validación con undersampling:", len(idx_valid))

meta_valid = metadatos.loc[idx_valid, ["numero_de_cliente", "foto_mes", "clase_ternaria", "clase01"]].reset_index(drop=True)


Train con undersampling: 306255
Validación con undersampling: 131152


In [9]:
def preparar_datos(indices_train, indices_eval, semilla, parametros):
    numerico = TransformToNumerical()
    X_numerico = numerico.fit_transform(X_baseline[indices_train])
    y = metadatos.loc[indices_train, "clase01"].to_numpy()
    generador = EnsembleGenerator(
        classification=True,
        n_estimators=parametros["fijos"]["n_estimators_finetune"],
        random_state=semilla, **parametros["fijos"]["preprocesamiento"],
    )
    generador.fit(X_numerico, y)
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    X_train = np.ascontiguousarray(generador.preprocessors_[metodo].X_transformed_, dtype=np.float32)
    generador.X_ = generador.y_ = None
    generador.preprocessors_[metodo].X_transformed_ = None
    del X_numerico
    gc.collect()

    X_eval = np.empty((len(indices_eval), X_train.shape[1]), dtype=np.float32)
    bloque = parametros["ejecucion"]["tamanos_consulta_oom"][0]
    for desde in range(0, len(indices_eval), bloque):
        hasta = min(desde + bloque, len(indices_eval))
        entrada = numerico.transform(X_baseline[indices_eval[desde:hasta]])
        X_eval[desde:hasta] = generador.transform(entrada, mode="test")[metodo][0][0]
    assert np.isfinite(X_train).all() and np.isfinite(X_eval).all()
    return X_train, y, X_eval


In [10]:
X_train, y_train, X_valid = preparar_datos(idx_train, idx_valid, SEED, PARAM)
y_valid = meta_valid["clase01"].to_numpy()
print("Training/contexto fijo:", X_train.shape, "| validación fija:", X_valid.shape)


Training/contexto fijo: (306255, 762) | validación fija: (131152, 762)


## 6. TFM congelado, adaptador y predicción


In [11]:
fijos = PARAM["fijos"]
trainer = FinetunedTabICLClassifier(
    device=str(DEVICE), random_state=SEED,
    n_estimators_finetune=fijos["n_estimators_finetune"],
    n_estimators_inference=fijos["n_estimators_inference"],
    freeze_col=fijos["freeze_col"], freeze_row=fijos["freeze_row"], freeze_icl=fijos["freeze_icl"],
    **fijos["preprocesamiento"],
)
trainer._load_pretrained(DEVICE)
modelo = trainer.model_.to(DEVICE)
trainer._apply_freezing(modelo)
assert all(not p.requires_grad for p in modelo.parameters())


class CapaCross(torch.nn.Module):
    def __init__(self, dimension, rango, hp, parametros):
        super().__init__()
        self.reducir = torch.nn.Linear(dimension, rango, bias=False)
        self.expandir = torch.nn.Linear(rango, dimension)
        self.normalizar = torch.nn.BatchNorm1d(dimension) if hp["batchnorm"] else torch.nn.Identity()
        inicializar = getattr(torch.nn.init, parametros["inicializacion"] + "_")
        inicializar(self.reducir.weight, std=parametros["std_inicial"])
        inicializar(self.expandir.weight, std=parametros["std_inicial"])
        torch.nn.init.constant_(self.expandir.bias, parametros["sesgo_inicial"])

    def forward(self, x0, x):
        return x + self.normalizar(x0 * self.expandir(self.reducir(x)))


class AdaptadorCross(torch.nn.Module):
    def __init__(self, dimension, hp, parametros):
        super().__init__()
        assert parametros["tipo"] == "cross"
        rango = max(1, int(dimension * parametros["rango_relativo"]))
        self.capas = torch.nn.ModuleList([
            CapaCross(dimension, rango, hp, parametros) for _ in range(parametros["capas"])
        ])
        self.alpha = torch.nn.Parameter(torch.full((dimension,), hp["compuerta_inicial"]))

    def forward(self, X):
        # El adaptador calcula en FP32; el TFM conserva el AMP del training oficial.
        with torch.autocast(device_type=X.device.type, enabled=False):
            x0 = X.float().reshape(-1, X.shape[-1])
            x = x0
            for capa in self.capas:
                x = capa(x0, x)
            return (x0 + self.alpha * (x - x0)).reshape(X.shape)


In [12]:
class EntradaPreparada:
    def transform(self, X):
        return np.asarray(X)


class VistaPreparada:
    def __init__(self, dimension, metodo):
        self.metodo = metodo
        self.feature_shuffles_ = {metodo: [list(range(dimension))]}
        self.class_shuffles_ = {metodo: [[0, 1]]}

    def transform(self, X, mode="test"):
        return OrderedDict([(self.metodo, (np.asarray(X)[None],))])


def compactar_kv(modulo, entrada, salida):
    if isinstance(salida, tuple):
        kv = [t.clone(memory_format=torch.contiguous_format)
              if t.untyped_storage().nbytes() > t.numel() * t.element_size() else t
              for t in salida[1:]]
        torch.cuda.synchronize(DEVICE)
        return (salida[0], *kv)
    torch.cuda.synchronize(DEVICE)


def preparar_cache(estimador, X, y, metodo):
    estimador._resolve_device()
    estimador._build_inference_config()
    estimador.n_samples_in_, estimador.n_features_in_ = X.shape
    estimador.y_encoder_ = LabelEncoder().fit(y)
    estimador.classes_ = estimador.y_encoder_.classes_
    estimador.n_classes_ = len(estimador.classes_)
    estimador.X_encoder_ = EntradaPreparada()
    estimador.ensemble_generator_ = VistaPreparada(X.shape[1], metodo)
    estimador.cache_mode_ = estimador.kv_cache
    entrada = torch.from_numpy(X[None]).float().to(DEVICE)
    etiquetas = torch.tensor(y[None], dtype=torch.float32, device=DEVICE)
    try:
        with ExitStack() as pila:
            # Conservar las fronteras sincronizadas de la evaluación que funcionó.
            pila.callback(modelo.row_interactor.register_forward_hook(
                lambda *args: torch.cuda.synchronize(DEVICE)).remove)
            for bloque in modelo.icl_predictor.tf_icl.blocks:
                pila.callback(bloque.register_forward_pre_hook(
                    lambda *args: torch.cuda.synchronize(DEVICE)).remove)
                pila.callback(bloque.register_forward_hook(compactar_kv).remove)
            modelo.forward_with_cache(
                X_train=entrada, y_train=etiquetas,
                store_cache=True, cache_mode=estimador.cache_mode_,
                inference_config=estimador.inference_config_,
            )
    finally:
        del entrada, etiquetas
    estimador.model_kv_cache_ = OrderedDict([(metodo, modelo._cache)])
    modelo.clear_cache()


In [13]:
import traceback
from tqdm.auto import tqdm


def adaptar_numpy(X, adaptador):
    if adaptador is None:
        return X
    with torch.no_grad():
        entrada = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
        salida = adaptador(entrada).float().cpu().numpy()
    assert salida.shape == X.shape
    if not np.isfinite(salida).all():
        raise FloatingPointError("El adaptador produjo valores no finitos al evaluar.")
    return salida


@torch.no_grad()
def predecir(X_contexto, y_contexto, X_consultas, adaptador, parametros):
    e = parametros["ejecucion"]
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    tamanos = e["tamanos_consulta_oom"]
    modo_tfm = modelo.training
    modo_adaptador = adaptador.training if adaptador is not None else None
    modelo.eval()
    if adaptador is not None:
        adaptador.eval()
        adaptador.zero_grad(set_to_none=True)
    modelo.clear_cache()
    modelo.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    contexto = estimador = None
    raiz = Path(e["offload_raiz"]).expanduser()
    raiz.mkdir(parents=True, exist_ok=True)
    try:
        with tempfile.TemporaryDirectory(dir=raiz) as temporal:
            comun = {
                "use_amp": False, "use_fa3": e["use_fa3"],
                "safety_factor": e["safety_factor"], "use_async": e["use_async"],
            }
            inferencia = {
                nombre: dict(comun)
                for nombre in ("COL_CONFIG", "ROW_CONFIG", "ICL_CONFIG")
            }
            inferencia["COL_CONFIG"].update(
                cpu_safety_factor=e["cpu_safety_factor"],
                max_pinned_memory_mb=e["max_pinned_memory_mb"],
                disk_offload_dir=temporal,
            )
            estimador = trainer._build_inner_estimator(
                modelo, parametros["fijos"]["n_estimators_inference"], DEVICE
            )
            estimador.set_params(
                kv_cache=e["kv_cache"], batch_size=e["batch_size_api"],
                use_amp=False, use_fa3=e["use_fa3"],
                disk_offload_dir=temporal, inference_config=inferencia,
            )
            with torch.autocast(device_type=DEVICE.type, enabled=False):
                contexto = X_contexto
                if adaptador is not None:
                    contexto = np.empty_like(X_contexto)
                    for desde in tqdm(
                        range(0, len(contexto), tamanos[0]),
                        desc="Adaptando contexto", unit="bloque",
                    ):
                        hasta = min(desde + tamanos[0], len(contexto))
                        contexto[desde:hasta] = adaptar_numpy(X_contexto[desde:hasta], adaptador)

                print(f"Creando caché del contexto: {len(contexto):,} filas...", flush=True)
                inicio = time.perf_counter()
                preparar_cache(estimador, contexto, y_contexto, metodo)
                torch.cuda.synchronize(DEVICE)
                print(f"Caché lista en {(time.perf_counter()-inicio)/60:.2f} min.", flush=True)
                contexto = None
                proba = np.empty(len(X_consultas), dtype=np.float64)
                desde = indice_tamano = 0
                with tqdm(total=len(X_consultas), desc="Prediciendo", unit="filas") as progreso:
                    while desde < len(X_consultas):
                        hasta = min(desde + tamanos[indice_tamano], len(X_consultas))
                        try:
                            entrada = adaptar_numpy(X_consultas[desde:hasta], adaptador)
                            salida = estimador.predict_proba(entrada)
                            torch.cuda.synchronize(DEVICE)
                            if not np.isfinite(salida).all():
                                raise FloatingPointError(f"Probabilidades no finitas en filas {desde}:{hasta}.")
                            proba[desde:hasta] = salida[:, 1]
                        except torch.OutOfMemoryError as error:
                            traceback.clear_frames(error.__traceback__)
                            modelo.clear_cache()
                            gc.collect()
                            torch.cuda.empty_cache()
                            indice_tamano += 1
                            if indice_tamano == len(tamanos):
                                raise
                            tqdm.write(f"OOM de consulta; reintentando con {tamanos[indice_tamano]} filas.")
                            continue
                        progreso.update(hasta - desde)
                        desde = hasta
                        del entrada, salida
            return proba
    except BaseException as error:
        # No dejar temporales GPU en los frames de una ejecución fallida/interrumpida.
        traceback.clear_frames(error.__traceback__)
        raise
    finally:
        if estimador is not None:
            estimador.model_kv_cache_ = None
            estimador.ensemble_generator_ = None
        modelo.clear_cache()
        modelo.train(modo_tfm)
        if adaptador is not None:
            adaptador.train(modo_adaptador)
        del contexto, estimador
        gc.collect()
        torch.cuda.empty_cache()


## 7. Entrenamiento

Se conserva el entrenamiento de la notebook de referencia: solo se actualiza el adaptador. Inicialización, shuffle y particiones contexto/consulta dependen de la semilla de cada ejecución. El contexto completo de evaluación conserva siempre las mismas filas y etiquetas.

La cantidad de épocas ejecutadas sale del registro elegido; el horizonte del coseno se recupera de su configuración original. Así, el caso elegido en época 10 conserva el coseno de 30 épocas. No hay evaluaciones intermedias.


In [14]:
def entrenar(X, y, parametros, semilla, epochs):
    f = parametros["fijos"]
    hp = parametros["hp"]
    opt = f["optimizacion"]
    assert opt["scheduler"] == "coseno"
    fijar_semilla(semilla)
    adaptador = AdaptadorCross(X.shape[1], hp, f["adaptador"]).to(DEVICE)
    grupos = [
        {"params": [p for n, p in adaptador.named_parameters() if n != "alpha" and p.ndim >= 2]},
        {"params": [p for n, p in adaptador.named_parameters() if n != "alpha" and p.ndim < 2],
         "weight_decay": opt["weight_decay_otros"]},
        {"params": [adaptador.alpha], "lr": hp["lr_maximo"] * hp["factor_lr_compuerta"],
         "weight_decay": opt["weight_decay_compuerta"]},
    ]
    optimizador = getattr(torch.optim, opt["optimizador"])(grupos, lr=hp["lr_maximo"])
    _, scaler, amp_context = trainer._make_amp(DEVICE)
    n_clases = len(np.unique(y))
    pasos_epoca = math.ceil(len(y) / f["filas_por_bloque"])
    pasos_totales = f["epochs"] * pasos_epoca
    pasos_warmup = opt["warmup_epochs"] * pasos_epoca

    def factor_lr(paso):
        if paso < pasos_warmup:
            return paso / max(1, pasos_warmup)
        progreso = (paso - pasos_warmup) / max(1, pasos_totales - pasos_warmup)
        return max(0.0, (1 + math.cos(math.pi * progreso)) / 2)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)
    for epoca in range(1, epochs + 1):
        fijar_semilla(semilla + epoca)
        trainer._set_training_mode(True)
        adaptador.train()
        modelo.clear_cache()
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)
        inicio = time.perf_counter()
        orden = np.random.default_rng(np.random.SeedSequence([semilla, epoca])).permutation(len(y))
        suma_loss = vistas = updates = 0
        for numero, posiciones in enumerate(np.array_split(orden, pasos_epoca), 1):
            etiquetas = y[posiciones]
            contexto, consultas = _split_ctx_query(
                etiquetas, query_size=max(2, int(len(posiciones) * trainer.finetune_ctx_query_ratio)),
                seed=semilla + epoca + (numero - 1) * 7919, stratify=True,
            )
            indices = np.concatenate([contexto, consultas])
            batch = MetaBatch(
                X=torch.from_numpy(np.ascontiguousarray(X[posiciones[indices]])).unsqueeze(0),
                y_train=torch.from_numpy(etiquetas[contexto].astype(np.float32)).unsqueeze(0),
                y_query=torch.from_numpy(etiquetas[consultas]).unsqueeze(0), train_size=len(contexto),
            )
            assert not trainer._task_skip_batch(batch), "Consulta con clase ausente del contexto."
            batch = move_meta_batch(batch, DEVICE)
            optimizador.zero_grad(set_to_none=True)
            with amp_context():
                logits = modelo(adaptador(batch.X), batch.y_train)
                loss = torch.nn.functional.cross_entropy(
                    logits[..., :n_clases].reshape(-1, n_clases), batch.y_query.reshape(-1)
                )
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f"Loss no finita en época {epoca}, bloque {numero}.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(adaptador.parameters(), trainer.grad_clip)
            escala = scaler.get_scale()
            scaler.step(optimizador)
            scaler.update()
            scheduler.step()
            updates += int(scaler.get_scale() >= escala)
            vistas += len(posiciones)
            suma_loss += loss.item()
            del batch, logits, loss
            if numero == 1 or numero % parametros["ejecucion"]["log_cada_bloques"] == 0 or numero == pasos_epoca:
                print(f"Época {epoca}/{epochs} | bloque {numero}/{pasos_epoca} | "
                      f"loss={suma_loss/numero:.4f} | LR={optimizador.param_groups[0]['lr']:.2e}", flush=True)
        assert vistas == len(y) and updates > 0
        torch.cuda.synchronize(DEVICE)
        print(f"Época {epoca}: {(time.perf_counter()-inicio)/60:.2f} min | "
              f"updates={updates}/{pasos_epoca} | "
              f"pico VRAM={torch.cuda.max_memory_reserved(DEVICE)/2**30:.2f} GiB", flush=True)
        optimizador.zero_grad(set_to_none=True)
        modelo.clear_cache()
        gc.collect()
        torch.cuda.empty_cache()
    return adaptador


## 8. Resultados y ganancias

Los nombres de los cuatro CSV están en la primera celda. Todos quedan en el directorio del experimento. Al iniciar una nueva ejecución de la sección 9, las salidas existentes se renombran con fecha y hora. **El log de búsqueda no se modifica.**

Se guardan resultados después de cada semilla evaluada. Las fallas numéricas o de memoria se registran y se continúa con la siguiente ejecución; otros errores detienen la notebook. No se guardan pesos ni cachés persistentes.


In [15]:
def ganancias_por_corte(proba):
    orden = np.argsort(-proba, kind="stable")
    ganancia = np.where(meta_valid["clase_ternaria"].to_numpy()[orden] == "BAJA+2", 1072500, -27500)
    acumulada = np.cumsum(ganancia)
    cortes = np.array(sorted(set(PARAM["training"]["cortes"])), dtype=int)
    assert ((cortes > 0) & (cortes <= len(acumulada))).all(), "Hay cortes fuera del tamaño de validación."
    tabla = pd.DataFrame({"envios": cortes, "ganancia": acumulada[cortes - 1]})
    posicion = int(np.argmax(acumulada))
    return tabla, int(acumulada[posicion]), posicion + 1


def guardar_resultados():
    prob_valid.to_csv(ARCHIVO_PROBABILIDADES, index=False)
    pd.DataFrame(resultados).to_csv(ARCHIVO_RESULTADOS, index=False)
    ganancias = (pd.concat(tablas_ganancia, ignore_index=True) if tablas_ganancia
                 else pd.DataFrame(columns=["modelo", "semilla", "envios", "ganancia"]))
    ganancias.to_csv(ARCHIVO_GANANCIAS, index=False)
    # Solo promediar un caso cuando terminaron sus tres semillas, sin esconder fallas.
    completas = [c["modelo"] for c in CASOS if sum(
        r["modelo"] == c["modelo"] and r["estado"] == "ok" for r in resultados
    ) == len(SEMILLAS)]
    promedios = (ganancias.loc[ganancias["modelo"].isin(completas)]
                .groupby(["modelo", "envios"], as_index=False)["ganancia"].mean())
    promedios.to_csv(ARCHIVO_PROMEDIOS, index=False)


## 9. Ejecutar los dos casos × tres semillas


In [16]:
marca = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
for archivo in ARCHIVOS_SALIDA:
    if archivo.exists():
        archivo.rename(archivo.with_name(f"{archivo.stem}_{marca}{archivo.suffix}"))
print("Salidas:")
for archivo in ARCHIVOS_SALIDA:
    print(archivo)

prob_valid = meta_valid[["numero_de_cliente", "foto_mes", "clase_ternaria"]].copy()
resultados, tablas_ganancia = [], []
numero_ejecucion = 0
for caso in CASOS:
    for semilla in SEMILLAS:
        numero_ejecucion += 1
        parametros = deepcopy(caso["parametros"])
        print(f"\nEjecución {numero_ejecucion}/{len(CASOS)*len(SEMILLAS)} | "
              f"{caso['modelo']} | semilla={semilla} | época final={caso['epoch']} | "
              f"HP={parametros['hp']}", flush=True)
        registro = {
            "modelo": caso["modelo"], "semilla": semilla, "semilla_datos": SEED,
            "iter_log": caso["iter_log"], "fecha_log": caso["fecha_log"],
            "epoch": caso["epoch"], "auc_log": caso["auc_log"], **parametros["hp"],
            "estado": "en_curso", "error": "",
            "parametros": json.dumps(parametros, ensure_ascii=False),
            "inicio": datetime.now().isoformat(timespec="seconds"),
        }
        adaptador = proba = None
        inicio_train = time.perf_counter()
        try:
            adaptador = entrenar(X_train, y_train, parametros, semilla, caso["epoch"])
            registro["segundos_training"] = time.perf_counter() - inicio_train
            inicio_eval = time.perf_counter()
            proba = predecir(X_train, y_train, X_valid, adaptador, parametros)
            registro["segundos_evaluacion"] = time.perf_counter() - inicio_eval
            tabla, ganancia_maxima, envios_maximo = ganancias_por_corte(proba)
            registro.update(estado="ok", auc_validacion=roc_auc_score(y_valid, proba),
                            ganancia_maxima=ganancia_maxima, envios_maximo=envios_maximo)
            prob_valid[f"{caso['modelo']}_S_{semilla}"] = proba
            tabla.insert(0, "semilla", semilla)
            tabla.insert(0, "modelo", caso["modelo"])
            tablas_ganancia.append(tabla)
            print(f"Evaluación final | {caso['modelo']} | semilla={semilla} | "
                  f"AUC={registro['auc_validacion']:.6f} | ganancia máxima={ganancia_maxima:,}", flush=True)
        except (FloatingPointError, torch.OutOfMemoryError) as error:
            registro.update(estado="error", error=str(error))
            traceback.clear_frames(error.__traceback__)
            print(f"FALLÓ {caso['modelo']} semilla={semilla}: {error}. Se continúa con la siguiente.", flush=True)
        finally:
            registro["fin"] = datetime.now().isoformat(timespec="seconds")
            registro["segundos_total"] = time.perf_counter() - inicio_train
            del adaptador, proba
            modelo.clear_cache()
            gc.collect()
            torch.cuda.empty_cache()
        resultados.append(registro)
        guardar_resultados()

display(pd.DataFrame(resultados).drop(columns="parametros"))


Salidas:
/home/marco/code/DMEyF/exp/TabICL_Retouche/validacion_probabilidades_semillas.csv
/home/marco/code/DMEyF/exp/TabICL_Retouche/validacion_resultados_semillas.csv
/home/marco/code/DMEyF/exp/TabICL_Retouche/validacion_ganancias_semillas.csv
/home/marco/code/DMEyF/exp/TabICL_Retouche/validacion_ganancias_promedio.csv

Ejecución 1/6 | M1 | semilla=230047 | época final=10 | HP={'lr_maximo': 0.0001, 'factor_lr_compuerta': 3.0, 'compuerta_inicial': 0.02, 'batchnorm': True}
Época 1/10 | bloque 1/599 | loss=0.2566 | LR=1.67e-07


/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/native/transformers/cuda/attention_backward.cu:102.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra

Época 1/10 | bloque 20/599 | loss=0.2031 | LR=3.34e-06
Época 1/10 | bloque 40/599 | loss=0.2024 | LR=6.68e-06
Época 1/10 | bloque 60/599 | loss=0.2005 | LR=1.00e-05
Época 1/10 | bloque 80/599 | loss=0.1978 | LR=1.34e-05
Época 1/10 | bloque 100/599 | loss=0.1956 | LR=1.67e-05
Época 1/10 | bloque 120/599 | loss=0.1952 | LR=2.00e-05
Época 1/10 | bloque 140/599 | loss=0.1979 | LR=2.34e-05
Época 1/10 | bloque 160/599 | loss=0.1976 | LR=2.67e-05
Época 1/10 | bloque 180/599 | loss=0.1967 | LR=3.01e-05
Época 1/10 | bloque 200/599 | loss=0.1964 | LR=3.34e-05
Época 1/10 | bloque 220/599 | loss=0.1949 | LR=3.67e-05
Época 1/10 | bloque 240/599 | loss=0.1945 | LR=4.01e-05
Época 1/10 | bloque 260/599 | loss=0.1936 | LR=4.34e-05
Época 1/10 | bloque 280/599 | loss=0.1926 | LR=4.67e-05
Época 1/10 | bloque 300/599 | loss=0.1925 | LR=5.01e-05
Época 1/10 | bloque 320/599 | loss=0.1924 | LR=5.34e-05
Época 1/10 | bloque 340/599 | loss=0.1914 | LR=5.68e-05
Época 1/10 | bloque 360/599 | loss=0.1911 | LR=6.01e

/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Época 1/10 | bloque 20/599 | loss=0.2081 | LR=3.34e-06
Época 1/10 | bloque 40/599 | loss=0.2080 | LR=6.68e-06
Época 1/10 | bloque 60/599 | loss=0.1963 | LR=1.00e-05
Época 1/10 | bloque 80/599 | loss=0.1967 | LR=1.34e-05
Época 1/10 | bloque 100/599 | loss=0.1961 | LR=1.67e-05
Época 1/10 | bloque 120/599 | loss=0.1965 | LR=2.00e-05
Época 1/10 | bloque 140/599 | loss=0.1953 | LR=2.34e-05
Época 1/10 | bloque 160/599 | loss=0.1939 | LR=2.67e-05
Época 1/10 | bloque 180/599 | loss=0.1941 | LR=3.01e-05
Época 1/10 | bloque 200/599 | loss=0.1945 | LR=3.34e-05
Época 1/10 | bloque 220/599 | loss=0.1942 | LR=3.67e-05
Época 1/10 | bloque 240/599 | loss=0.1947 | LR=4.01e-05
Época 1/10 | bloque 260/599 | loss=0.1944 | LR=4.34e-05
Época 1/10 | bloque 280/599 | loss=0.1944 | LR=4.67e-05
Época 1/10 | bloque 300/599 | loss=0.1933 | LR=5.01e-05
Época 1/10 | bloque 320/599 | loss=0.1928 | LR=5.34e-05
Época 1/10 | bloque 340/599 | loss=0.1923 | LR=5.68e-05
Época 1/10 | bloque 360/599 | loss=0.1923 | LR=6.01e

Adaptando contexto:   0%|          | 0/599 [00:00<?, ?bloque/s]

... (20 outputs left out)

Creando caché del contexto: 306,255 filas...
Caché lista en 12.77 min.


Prediciendo:   0%|          | 0/131152 [00:00<?, ?filas/s]

Evaluación final | M2 | semilla=230063 | AUC=0.939979 | ganancia máxima=4,398,955,000


,modelo,semilla,semilla_datos,iter_log,fecha_log,epoch,auc_log,lr_maximo,factor_lr_compuerta,compuerta_inicial,...,estado,error,inicio,fin,segundos_total,segundos_training,segundos_evaluacion,auc_validacion,ganancia_maxima,envios_maximo
0,M1,230047,230047,1,2026-11-13T11:31:07,10,0.941781,0.0001,3.0,0.02,...,error,"Loss no finita en época 6, bloque 5.",2026-11-13T18:29:34,2026-11-13T18:46:14,1000.020546,NaN,NaN,NaN,NaN,NaN
1,M1,230059,230047,1,2026-11-13T11:31:07,10,0.941781,0.0001,3.0,0.02,...,ok,,2026-11-13T18:46:14,2026-11-13T19:36:34,3019.505215,1994.193896,1025.165665,0.941168,4.413668e+09,31583.0
2,M1,230063,230047,1,2026-11-13T11:31:07,10,0.941781,0.0001,3.0,0.02,...,ok,,2026-11-13T19:36:34,2026-11-13T20:27:41,3066.514696,1995.087818,1071.347901,0.941655,4.403438e+09,32395.0
3,M2,230047,230047,2,2026-11-13T15:49:50,30,0.941609,0.0005,3.0,0.02,...,ok,,2026-11-13T20:27:41,2026-11-13T22:23:48,6966.949416,5970.847323,996.025350,0.941636,4.396425e+09,30650.0
4,M2,230059,230047,2,2026-11-13T15:49:50,30,0.941609,0.0005,3.0,0.02,...,ok,,2026-11-13T22:23:49,2026-11-14T00:21:48,7079.173955,5974.349578,1104.748827,0.941881,4.384435e+09,32086.0
5,M2,230063,230047,2,2026-11-13T15:49:50,30,0.941609,0.0005,3.0,0.02,...,ok,,2026-11-14T00:21:48,2026-11-14T02:18:53,7024.509269,5991.035218,1033.399692,0.939979,4.398955e+09,32278.0


## 10. Ganancia promedio por corte

Cada línea es la media de las ganancias obtenidas por las tres semillas del caso. No se promedian probabilidades ni se construye un ensamble. El gráfico usa los cortes definidos en el log. Click en la leyenda permite ocultar una línea; doble click permite aislarla.


In [17]:
promedios = pd.read_csv(ARCHIVO_PROMEDIOS)
fig = go.Figure()
for caso in CASOS:
    tabla = promedios.loc[promedios["modelo"] == caso["modelo"]].sort_values("envios")
    if tabla.empty:
        print(f"{caso['modelo']}: faltan semillas completas; no se grafica un promedio parcial.")
        continue
    nombre = f"{caso['modelo']} | LR={caso['parametros']['hp']['lr_maximo']} | época={caso['epoch']}"
    fig.add_trace(go.Scatter(x=tabla["envios"], y=tabla["ganancia"] / 1e6,
                             mode="lines+markers", name=nombre))
fig.update_layout(title="Ganancia promedio de tres semillas — validación fija",
                  xaxis_title="Cantidad de envíos", yaxis_title="Ganancia promedio (millones)")
fig.show()
display(promedios.pivot(index="envios", columns="modelo", values="ganancia"))


M1: faltan semillas completas; no se grafica un promedio parcial.


modelo,M2
envios,
9000,2.840933e+09
9500,2.959917e+09
10000,3.063867e+09
10500,3.164517e+09
11000,3.255267e+09
11500,3.348583e+09
12000,3.427233e+09
12500,3.509550e+09
13000,3.577200e+09
